# kidspipe — DR5 Exact Continuation (Colab CLI / Option B)

Cloned from `Recreate_KiDS-pipeline.ipynb` to run headlessly via the `colab` CLI (`colab exec` / `colab ssh`) without requiring an open browser tab or interactive `drive.mount`.

- **Repo:** `SangkaAnut/kidspipe` (`main`) at `/content/kidspipe`
- **Option B state directory:** `/content/kidspipe_state` (synced to/from `/MyDrive/kidspipe` via `gdrive` + `colab ssh`)

1. Run **Cell 1 (`1. Session setup`)** to configure `/content/kidspipe` and `KIDSPIPE_*` paths.
2. Run **Cell 2** to confirm repo status and environment variables.
3. Run **Cell 3** and **Cell 5** (or launch `dr5_chain.py exact` via `colab ssh`) to continue the remaining `exact` pairs.


In [ ]:
#@title 1. Session setup (run this first, in EVERY session) { display-mode: "form" }
#@markdown Private repository to sync (`owner/name`); uses `GH_TOKEN` env var, `gh` credentials, or Colab Secret `GH_TOKEN`.
GITHUB_REPO = "SangkaAnut/kidspipe"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Option B local VM state directory (synced to/from Drive via gdrive + colab ssh); set MOUNT_DRIVE=True only in interactive browser sessions.
DRIVE_ROOT = "/content/kidspipe_state"  #@param {type:"string"}
MOUNT_DRIVE = False  #@param {type:"boolean"}
CATALOGUE_DIR = "/content/dr5"  #@param {type:"string"}
REPO_DIR = "/content/kidspipe"

import base64, importlib, os, pathlib, shutil, subprocess, sys

def _git(args, token=None, cwd=None):
    """Run git with optional token HTTP header (not stored in .git/config); masks token in output."""
    cmd, secret = ["git"], []
    if token:
        b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {b64}"]
        secret = [token, b64]
    r = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secret:
        out = out.replace(s, "***")
    return r.returncode, out.strip()

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab: Drive is not mounted; DRIVE_ROOT must be a writable directory.")

token = os.environ.get("GH_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception as e:
        print(f"No GH_TOKEN secret available ({type(e).__name__}); using gh git credentials, existing clone, or Drive zip.")

repo = pathlib.Path(REPO_DIR)
url = f"https://github.com/{GITHUB_REPO}.git"
if (repo / "pyproject.toml").exists():
    _, dirty = _git(["status", "--porcelain"], cwd=repo)
    if dirty:
        print("Local uncommitted changes detected in /content/kidspipe; skipping pull to preserve local edits.")
    else:
        rc, out = _git(["pull", "--ff-only", "origin", BRANCH], token, cwd=repo)
        if rc == 0:
            print("repository synced to latest", BRANCH)
        else:
            print("could not update (using existing clone):", out[-300:])
elif token or not pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip").exists():
    rc, out = _git(["clone", "--branch", BRANCH, url, REPO_DIR], token)
    if rc != 0:
        raise SystemExit("git clone failed (check gh auth or GH_TOKEN secret): " + out[-500:])
    print("cloned", GITHUB_REPO)
else:
    tmp = pathlib.Path("/content/_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip")), str(tmp))
    top = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(top[0]) if len(top) == 1 and not (tmp / "pyproject.toml").exists() else str(tmp), REPO_DIR)
    print("unpacked the repository from the Drive zip")

sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
import setup_env
importlib.reload(setup_env)
INFO = setup_env.setup(REPO_DIR, DRIVE_ROOT, CATALOGUE_DIR, numba_cache="local")
os.chdir(REPO_DIR)
WORKERS = os.cpu_count() or 1


In [ ]:
#@title 2. Check repo sync & kidspipe environment
import os, subprocess
print(subprocess.check_output(["git", "-C", "/content/kidspipe", "status", "-sb"], text=True).strip())
print("HEAD:", subprocess.check_output(["git", "-C", "/content/kidspipe", "log", "-1", "--oneline"], text=True).strip())
for k in ("KIDSPIPE_ROOT", "KIDSPIPE_RUNS", "KIDSPIPE_DR5_WORK", "KIDSPIPE_DR5_CATALOGUE", "KIDSPIPE_PLOTS", "NUMBA_CACHE_DIR", "OMP_NUM_THREADS", "NUMBA_NUM_THREADS"):
    print(f"{k:24s} {os.environ.get(k)}")

## 3. Weak-Lensing Physics & Mathematics: $\xi_{\pm}$, $m$-Bias, COSEBIs ($E_n, B_n$), and Validation Gates

This pipeline re-measures the cosmic-shear two-point statistics from the public **KiDS-Legacy (DR5)** gold shear catalogue ($40{,}894{,}394$ galaxies across $6$ tomographic redshift bins, giving $N_{\mathrm{pair}} = \frac{6 \times 7}{2} = 21$ bin pairs $1 \le a \le b \le 6$) and validates them against the official release products (`KiDS_Legacy_xipm.fits` and `KiDS_Legacy_cosebis.fits`).

### 3.1 Shear & Ellipticity Sky Convention
Observed galaxy ellipticities $e = e_1 + i e_2$ are defined in the local sky $(\mathrm{RA}, \mathrm{Dec})$ frame ($e_1$ along $+\mathrm{RA}$, $e_2$ at $+45^\circ$ toward $+\mathrm{Dec}$). Galaxy-galaxy lensing ($\gamma_t / \gamma_\times$) and parity checks (`SIGN_CONVENTION.md`) confirm that the DR5 catalogue ellipticities are used **unflipped** (`flip_g1 = flip_g2 = False`).

For a galaxy pair $(i, j)$ with separation angle $\theta_{ij}$ and position angle $\phi_{ij} = \operatorname{atan2}(\Delta\mathrm{Dec},\, \Delta\mathrm{RA}\cos\overline{\mathrm{Dec}})$, the tangential and cross shear components along the connecting vector are:
$$e_t + i e_\times = -e\,e^{-2i\phi_{ij}}.$$

### 3.2 Tomographic Two-Point Shear Correlation Functions ($\xi_+, \xi_-$)
For tomographic bins $a \le b$ and $1000$ log-spaced angular bins $\theta \in [2, 300]\,\mathrm{arcmin}$, the weighted estimators (unordered pairs for auto-correlations $a = b$; all pairs for cross-correlations $a < b$) are:
$$\hat{\xi}_+(\theta) = \frac{\sum_{i,j} w_i w_j\,\mathrm{Re}\!\left(e_i e_j^*\right)}{\sum_{i,j} w_i w_j} = \frac{\sum_{i,j} w_i w_j \left(e_{1,i} e_{1,j} + e_{2,i} e_{2,j}\right)}{\sum_{i,j} w_i w_j},$$
$$\hat{\xi}_-(\theta) + i\,\hat{\xi}_\times(\theta) = \frac{\sum_{i,j} w_i w_j\, e_i e_j\, e^{-4i\phi_{ij}}}{\sum_{i,j} w_i w_j} = \frac{\sum_{i,j} w_i w_j \left[(e_{t,i} e_{t,j} - e_{\times,i} e_{\times,j}) + i(e_{t,i} e_{\times,j} + e_{\times,i} e_{t,j})\right]}{\sum_{i,j} w_i w_j}.$$

In the flat-sky limit, $\xi_\pm(\theta)$ relate to the $E$- and $B$-mode convergence angular power spectra $C_E(\ell)$ and $C_B(\ell)$ via Hankel transforms:
$$\xi_+(\theta) = \frac{1}{2\pi}\int_0^\infty \mathrm{d}\ell\,\ell\,\left[C_E(\ell) + C_B(\ell)\right] J_0(\ell\theta), \qquad \xi_-(\theta) = \frac{1}{2\pi}\int_0^\infty \mathrm{d}\ell\,\ell\,\left[C_E(\ell) - C_B(\ell)\right] J_4(\ell\theta).$$

- **TreeCorr (`bin_slop = 1.5`, default `angle_slop = 0.1`):** matches the official KiDS-Legacy measurement configuration (Wright et al. 2025, Sect. 2.5.1) for a like-for-like comparison.
- **Independent KD-tree `PairCounter` (`kidspipe.measure.paircount`):** computes chord distances $d = 2\sin(\theta/2)$ on 3-D unit vectors and rotates each galaxy's shear individually by $e^{-2i\phi_{ij}}$ as a code-independent cross-check (Gate A4).

### 3.3 Multiplicative Bias ($m$), $n_{\mathrm{eff}}$, and Shape Noise ($\sigma_e$)
Catalogue `e1, e2` carry additive ($c$-term) calibration but **no** multiplicative shear bias correction. Using the per-bin multiplicative bias $m_a$ (Wright et al. 2025 Table 1: $m = [-0.023, -0.016, -0.011, +0.020, +0.030, +0.045]$):
$$\hat{\xi}_\pm^{\,\mathrm{corr}}(\theta) = \frac{\hat{\xi}_\pm^{\,\mathrm{raw}}(\theta)}{(1 + m_a)(1 + m_b)}.$$

Per-bin effective number density ($A_{\mathrm{eff}} = 967.4\,\mathrm{deg}^2$) and per-component ellipticity dispersion (Joachimi et al. 2021, Eq. C.9) are:
$$n_{\mathrm{eff}}^{(a)} = \frac{\left(\sum_i w_i\right)^2}{A_{\mathrm{eff}} \sum_i w_i^2}, \qquad \sigma_{e,a}^2 = \frac{\sum_i w_i^2 \left(e_{1,i}^2 + e_{2,i}^2\right)}{2\,(1 + m_a)^2 \sum_i w_i^2}.$$

For comparison with `KiDS_Legacy_xipm.fits`, the 1000 fine bins are rebinned into $9$ log-spaced angular bins on $[2, 300]\,\mathrm{arcmin}$ weighted by `npairs`.

### 3.4 COSEBIs ($E_n, B_n$) from Binned $\xi_\pm$
COSEBIs (Complete Orthogonal Sets of E/B-Integrals) cleanly separate $E$-modes (cosmological shear) and $B$-modes (systematics null test) on a finite interval $[\theta_{\min}, \theta_{\max}] = [2, 300]\,\mathrm{arcmin}$ ($z = \ln(\theta/\theta_{\min})$, $Z = \ln(\theta_{\max}/\theta_{\min})$):
$$E_n = \frac{1}{2}\int_{\theta_{\min}}^{\theta_{\max}} \mathrm{d}\theta\,\theta\left[T_{+n}(\theta)\,\xi_+(\theta) + T_{-n}(\theta)\,\xi_-(\theta)\right], \qquad B_n = \frac{1}{2}\int_{\theta_{\min}}^{\theta_{\max}} \mathrm{d}\theta\,\theta\left[T_{+n}(\theta)\,\xi_+(\theta) - T_{-n}(\theta)\,\xi_-(\theta)\right].$$

Here $T_{+n}(z) = N_n \prod_{i=1}^{n+1}(z - r_{n,i})$ satisfies the zero-moment constraints $\int_0^Z e^{2z} T_{+n}(z)\,\mathrm{d}z = \int_0^Z e^{4z} T_{+n}(z)\,\mathrm{d}z = 0$ and orthonormality $\int_{\theta_{\min}}^{\theta_{\max}} T_{+n} T_{+m}\,\mathrm{d}\theta = (\theta_{\max}-\theta_{\min})\delta_{nm}$, while
$$T_{-n}(z) = T_{+n}(z) + \int_0^z 4\,T_{+n}(y)\left[e^{2(y-z)} - 3\,e^{4(y-z)}\right]\mathrm{d}y.$$
On the 1000 fine bins ($\theta$ in radians), `e_from_xi_binned(..., scheme="edges", unit="rad")` integrates using exact bin-edge weights $w_k = \frac{1}{2}(b_{k+1}^2 - b_k^2)$.

### 3.5 Validation Gates Against the Official Release
- **Selection / $n_{\mathrm{eff}}$ / $\sigma_e$:** Per-bin galaxy counts match Wright+25 Table 1 `Ngold`; $n_{\mathrm{eff}}$ and $\sigma_e$ match published values.
- **A1b-noise (1000 fine bins vs official ASCII):** After fitting a per-pair amplitude ratio $A_p = \frac{\sum_k g_k \xi_k^{\mathrm{raw}}}{\sum_k g_k \xi_k^{\mathrm{ascii}}}$, the 20-bin boxcar residual RMS over white-noise expectation must satisfy $\le 1.03$ across all 42 curves, and $\ln A_p$ must be consistent with a constant ($\mathrm{PTE} > 0.01$).
- **A3 (9 official bins, $m$-corrected vs `KiDS_Legacy_xipm.fits`):**
  $$\max_{p,\theta} \frac{\left|\hat{\xi}_{\pm,p}^{\,\mathrm{corr}}(\theta) - \xi_{\pm,p}^{\,\mathrm{official}}(\theta)\right|}{\sigma_{\mathrm{official}}(\theta)} < 0.1, \qquad \chi^2_{378} = \Delta\boldsymbol{\xi}^T C_{\xi}^{-1} \Delta\boldsymbol{\xi}.$$
- **A4 (Independent estimator):** `PairCounter` vs TreeCorr (`bin_slop = 0`) on subsample.
- **B1–B3 & D (`KiDS_Legacy_cosebis.fits`):** Integration consistency ($E_n, B_n$ vs official), $B$-mode null test $\chi^2 = \mathbf{B}^T C_{BB}^{-1}\mathbf{B}$ (modes 1–6 and 1–20), and best-fit model $\chi^2$ (Step D).

## 4. End-to-End DR5 Measurement & Validation Chain

Runs `scripts/dr5_chain.py` across all stages (`download` $\to$ `extract` $\to$ `measure` 21 pairs at `bin_slop=1.5` $\to$ `eval` $\to$ `product` $\to$ `plots` $\to$ `compare`):
- **Catalogue (`download`):** 7.2 GB `.fits.gz` from the KiDS DR5 server $\to$ 16.4 GB uncompressed FITS on local disk (`/content/dr5/`).
- **Per-bin extract (`extract`):** 6 tomographic NPZ files (`bin0..5.npz`, ~1.6 GB) saved persistently to Google Drive (`/content/drive/MyDrive/kidspipe/dr5_work`).
- **Measurement (`measure`):** TreeCorr `bin_slop = 1.5` on all 21 pairs (~15.8 h on 8 vCPU, ~5 GB peak RAM per cross pair, resumable per pair to Google Drive).
- **Validation (`eval`, `product`, `plots`):** A1b-noise, A3 vs `KiDS_Legacy_xipm.fits`, A4 `PairCounter` re-check, B1–B3 & D vs `KiDS_Legacy_cosebis.fits`, and `dr5_01..06` figures.

In [ ]:
#@title 3. DR5 Session parameters & helpers { display-mode: "form" }
MAX_HOURS = 22  #@param {type:"number"}
#@markdown free Colab: ~11 h; Pro: up to ~23 h
DELETE_GZ = True  #@param {type:"boolean"}
LOCAL_WORK_CACHE = True  #@param {type:"boolean"}
#@markdown copy per-bin npz from Drive to local scratch once per session

import os, pathlib, sys
REPO_DIR = "/content/kidspipe"
if f"{REPO_DIR}/scripts/colab" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
from setup_env import sh, launch, tail, stop

W = globals().get("WORKERS", os.cpu_count() or 1)
CHAIN = "python3 scripts/dr5_chain.py"
CACHE = " --work-cache /content/dr5_work_local" if LOCAL_WORK_CACHE else ""
COMMON = f"--threads {W}"
LOG = pathlib.Path(os.environ.get("KIDSPIPE_RUNS", f"{REPO_DIR}/runs")) / "dr5_measure" / "chain_logs" / "session.log"

def run(sub, extra=""):
    rc = sh(f"{CHAIN} {sub} {COMMON} {extra}")
    print(f"\nexit status {rc}" + ("  -> not finished: re-run in a new session" if rc == 75 else "  (failed)" if rc else "  (complete)"))
    return rc

print(f"Configured: threads={W}, max_hours={MAX_HOURS}, log={LOG}")

In [ ]:
#@title 4. Check DR5 chain status
run("status")

In [ ]:
#@title 5. Launch DR5 exact continuation detached (resumable per chunk)
EXACT_PAIRS = "3-5,2-5,3-4,1-6,4-5,4-4,3-6,4-6,5-6,3-3,5-5,6-6"  #@param {type:"string"}
LOG = pathlib.Path(os.environ.get("KIDSPIPE_RUNS", f"{REPO_DIR}/runs")) / "dr5_measure" / "chain_logs" / "exact_cli.log"
pair_flag = f" --pairs {EXACT_PAIRS.strip()}" if EXACT_PAIRS.strip() else ""
flags = f"--max-hours {MAX_HOURS}{CACHE}{pair_flag}"
if "DR5_PROC" in globals() and DR5_PROC is not None and DR5_PROC.poll() is None:
    print(f"Job already running (PID {DR5_PROC.pid}); check Cell 6 or stop it in Cell 7 first.")
else:
    (LOG.parents[1] / "chain_STOP").unlink(missing_ok=True)
    cmd = f"{CHAIN} exact {COMMON} {flags}"
    DR5_PROC = launch(cmd, LOG)
    print(f"Launched detached PID {DR5_PROC.pid}")
    print(f"Command: {cmd}")
    print(f"Log: {LOG}")


In [ ]:
#@title 6. Monitor live log & DR5 chain status
if "DR5_PROC" in globals() and DR5_PROC is not None:
    rc = DR5_PROC.poll()
    print("Job state:", "RUNNING" if rc is None else f"FINISHED (exit={rc})")
print("--- log tail ---")
print(tail(LOG, 35))
print("\n--- status ---")
run("status")

In [ ]:
#@title 7. Stop detached job (optional) { display-mode: "form" }
STOP_JOB = False  #@param {type:"boolean"}
if STOP_JOB:
    if "DR5_PROC" in globals() and DR5_PROC is not None and DR5_PROC.poll() is None:
        stop(DR5_PROC)
        print(f"Sent SIGTERM to process group {DR5_PROC.pid}")
    else:
        print("No running DR5_PROC in this kernel.")

In [ ]:
#@title 8. Validate against official KiDS-Legacy release, compare & show plots (run after 21/21 pairs complete)
from IPython.display import Image, display

if run("eval") == 0:
    run("product")
    run("plots")
    run("compare")
    sh("python3 -m pytest -q tests/test_dr5_product.py -p no:cacheprovider")
    plots_dir = os.environ.get("KIDSPIPE_PLOTS", "/content/kidspipe/plots")
    for name in ("dr5_01_xipm_vs_official.png", "dr5_03_En_vs_official.png"):
        for label, d in (("yours", plots_dir), ("shipped", "/content/kidspipe/plots")):
            f = pathlib.Path(d) / name
            print(label, f, "(missing)" if not f.exists() else "")
            if f.exists():
                display(Image(filename=str(f), width=700))

In [ ]:
#@title 9. Show DR5 validation plots
import os, pathlib
from IPython.display import Image, display

plots_dir = pathlib.Path(os.environ.get("KIDSPIPE_PLOTS", "/content/drive/MyDrive/kidspipe/plots"))
pngs = sorted(plots_dir.glob("*.png"))
if not pngs:
    print(f"No .png plots found in {plots_dir}")
else:
    for p in pngs:
        print(p.name)
        display(Image(filename=str(p), width=800))

In [ ]:
#@title 10. Show exact xi_pm & COSEBIs plots (yours if generated, plus shipped reference)
import os, pathlib
from IPython.display import Image, display

plots_dir = pathlib.Path(os.environ.get("KIDSPIPE_PLOTS", "/content/drive/MyDrive/kidspipe/plots"))
shipped_dir = pathlib.Path("/content/kidspipe/plots")
exact_names = (
    "dr5_08_xipm_exact_vs_official.png",
    "dr5_09_En_exact.png",
    "dr5_09b_Bn_exact.png",
    "dr5_exact_vs_b1p5.png",
)
for name in exact_names:
    for label, d in (("yours", plots_dir), ("shipped", shipped_dir)):
        f = d / name
        if f.exists():
            print(f"{label}: {f}")
            display(Image(filename=str(f), width=800))
        else:
            print(f"{label}: {f} (not yet generated — run assemble, exact-product, and plots after all 21 exact pairs complete)")